<a href="https://colab.research.google.com/github/Lisura123/J26-IT-348/blob/component2-tinyml/component2-tinyml/notebooks/colab_quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Component 2: INT8 quantisation (Colab)

Derives the INT8 deployment version of the frozen model, following `docs/QUANTIZATION.md` (`int8-v1`).

- **Float32 preprocessing + INT8 neural inference.** The float contract in `models/` is not changed; the INT8 parameters are a deployment derivative.
- **TRAINING rows only.** OPERATION rows are dropped straight after loading and are never used.
- Nothing is trained or refitted. Thresholds, features and labels are unchanged.
- The host C tests are compiled with gcc in this session. **No ESP32 measurement is made**, so no memory, latency or energy result is claimed.

All code comes from `src/tinyml/` (`quantize.py`, `device_reference.py`, `golden.py`, `codegen.py`) and `scripts/07_quantize_model.py`. This notebook only calls it.

**Two manual safety settings:**
1. **Step 1:** paste the full 40-character approved commit hash into `APPROVED_COMMIT`. The notebook checks out exactly that commit (never a newer one) and checks that it belongs to `component2-tinyml`.
2. **Step 5:** quantisation runs only after you change `CONFIRM_QUANTIZATION = False` to `True` by hand. Step 5 also re-checks the commit.

Run the cells one at a time with **Shift + Enter** (not "Run all").

## Step 1: Get the approved code revision and install dependencies
Paste the **full 40-character commit hash** you were given for the official run into `APPROVED_COMMIT`, then run the cell. The cell checks out exactly that commit and stops if the hash is missing, malformed, unknown or not part of `component2-tinyml`. It never moves to a newer branch revision.

In [3]:
# STEP 1: Check out the approved revision and install dependencies
import os, re, sys, subprocess

APPROVED_COMMIT = "ef6e9ae9badeeb06b149bc1177d546d014d54ed2"   # paste the full 40-character approved commit hash here

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
    raise SystemExit("Paste the full 40-character approved commit hash into APPROVED_COMMIT first.")

def git(*args, check=True):
    return subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True, check=check)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
git("fetch", "origin", BRANCH)
if git("cat-file", "-e", f"{APPROVED_COMMIT}^{{commit}}", check=False).returncode != 0:
    raise SystemExit(f"Commit {APPROVED_COMMIT} was not found in the repository.")
if git("merge-base", "--is-ancestor", APPROVED_COMMIT, f"origin/{BRANCH}", check=False).returncode != 0:
    raise SystemExit(f"Commit {APPROVED_COMMIT} is not part of origin/{BRANCH}.")
git("checkout", "--quiet", "--detach", APPROVED_COMMIT)          # exactly this revision, never a newer one
HEAD = git("rev-parse", "HEAD").stdout.strip()
if HEAD != APPROVED_COMMIT or git("status", "--porcelain").stdout.strip():
    raise SystemExit(f"Checked-out revision {HEAD} is not the clean approved revision {APPROVED_COMMIT}.")
COMMIT = HEAD[:7]
print("Approved revision checked out:", git("log", "-1", "--format=%H %s").stdout.strip())
newer = git("rev-list", "--count", f"{APPROVED_COMMIT}..origin/{BRANCH}").stdout.strip()
print(f"Newer commits on origin/{BRANCH} (not used): {newer}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)
print(subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

Approved revision checked out: ef6e9ae9badeeb06b149bc1177d546d014d54ed2 Add Revision Pinning and Confirmation Guard to Quantization Notebook
Newer commits on origin/component2-tinyml (not used): 0
Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Step 2: Import the modules and verify the frozen model (no data is read)

In [4]:
# STEP 2: Import modules and verify the committed model artifacts
import importlib.util
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.train import check_dataset_sha256
from tinyml.holdout import verify_frozen_model
from tinyml.quantize import QUANT_VERSION

MODEL_DIR = f"{COMPONENT_DIR}/models"
contract = verify_frozen_model(MODEL_DIR)          # pinned SHA-256 values; stops on any difference
print("Quantisation:", QUANT_VERSION)
print("Model       :", contract["model_version"], "| status", contract["status"],
      "| weights", contract["weights_sha256"][:16], "...")
print("Frozen model verified.")

spec = importlib.util.spec_from_file_location("quant", f"{COMPONENT_DIR}/scripts/07_quantize_model.py")
quant = importlib.util.module_from_spec(spec); spec.loader.exec_module(quant)

Quantisation: int8-v1
Model       : c2-local-v1 | status TRAINED | weights 94e7ab9384a1906e ...
Frozen model verified.


## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. Only the TRAINING rows are used.

In [5]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

Saving pv_fault_dataset.csv to pv_fault_dataset.csv
Dataset file: /content/pv_fault_dataset.csv | 957366 bytes


## Step 4: Check the dataset fingerprint (stops if it is not the validated file)

In [6]:
# STEP 4: SHA-256 check
sha = check_dataset_sha256(file_sha256(DATA_PATH))
print("SHA-256 OK:", sha)

SHA-256 OK: 5fbb93471c056c274cbf81e8c317934e379b2484013147e4c2cacfa93421595b


## Step 5: Quantise (TRAINING rows only, once)
Calibrates on status-OK TRAINING rows, compares INT8 with the float reference, builds the golden vectors, generates the C headers and runs the host C tests. Writes everything to `/content/quant_out/` and refuses to overwrite earlier output.

**This cell refuses to run until you change `CONFIRM_QUANTIZATION = False` to `True` by hand.** It also re-checks that the approved commit from Step 1 is still checked out. Run it **once**.

In [7]:
# STEP 5: INT8 quantisation (no OPERATION data) - official run, once
CONFIRM_QUANTIZATION = True   # change to True by hand for the official run

if not CONFIRM_QUANTIZATION:
    raise SystemExit("Set CONFIRM_QUANTIZATION = True in this cell to run the official quantisation.")
head_now = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                          capture_output=True, text=True, check=True).stdout.strip()
if head_now != APPROVED_COMMIT:
    raise SystemExit(f"Checked-out revision {head_now} is not the approved {APPROVED_COMMIT}. Re-run Step 1.")

SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
OUT_ROOT = "/content/quant_out"
report, q, g = quant.run(DATA_PATH, SPLITS_PATH, MODEL_DIR, OUT_ROOT, code_commit=COMMIT)
a = report["acceptance"]
for k, v in a["checks"].items():
    print(f"{k:28s} {'PASS' if v else 'FAIL'}")
print("INT8 accepted:", a["int8_accepted"], "|", a["deployment_candidate"])
print("Host C tests :", report["host_c_tests"]["output"][-120:])

safety_changes_zero          PASS
agreement_at_least_0.995     PASS
macro_f1_drop_at_most_1pp    PASS
golden_tests_passed          PASS
INT8 accepted: True | INT8 and Float32 (final choice by on-device measurement)
Host C tests : ALL PASSED 2920


## Step 6: Show the report

In [8]:
# STEP 6: Show the Markdown report
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT_ROOT}/results/quantization_report.md", encoding="utf-8").read()))

# INT8 quantisation report (TRAINING window only)

Quantisation `int8-v1`, created 2026-10-16T19:46:32Z. Model `c2-local-v1`, contract SHA-256 `752f62285cca62b59a629686fd5b941b748a811c78808751c04595ee3452cb2a`, weights SHA-256 `94e7ab9384a1906ec2cf0d3176b435e700f15cd99552fbc6c71519c648ecc3a7`. Code `ef6e9ae`. Environment {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1'}.

System: **Float32 preprocessing + INT8 neural inference**. The float contract is unchanged; the INT8 parameters are a deployment derivative. **OPERATION was not used.**

## Scales

A_x = 4.488935, s_x = 0.03534594, s_W1 = 0.01053759, s_W2 = 0.00879529, A_z1 = 6.152358, s_z1 = 0.04844377, s_h = 1/127; M1 = 2113405489, SH1 = 38; INV_SX = 0x1.c4ab2e0000000p+4, OUT_SCALE = 0x1.2279260000000p-14.

Accumulator bounds: {'acc1': 71507, 'acc2': 165764, 'acc1_times_m1': 151123286301923} (int32 limit 2147483647; int64 rescaling limit 2^62).

## Float32 device preprocessing check

{'status_identical': True, 'max_abs_z_difference': 3.039836883544922e-06, 'ok_rows': 2924, 'skipped_rows': 76}

## INT8 vs float reference (TRAINING status-OK rows)

Rows 2924, agreement 1.0000, changed predictions 0, changes involving OPEN/SHORT 0 {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}.

Change pairs: none. Per-class agreement: {'NORMAL': 1.0, 'PARTIAL_SHADING': 1.0, 'DUST': 1.0, 'OPEN_CIRCUIT': 1.0, 'SHORT_CIRCUIT': 1.0}.

Training-fit macro-F1: float 1.0000, INT8 1.0000, drop 0.0000.

Input clips 0 (rows 0); tanh lookup clips 0.

Float32 device path vs float reference: agreement 1.0000, changes 0.

## Golden vectors and host C tests

{'z_cases': {'TRAINING_DERIVED': 100, 'TRAINING_DERIVED_REFERENCE_PROBE': 5}, 'raw_cases': {'SYNTHETIC_PROTOTYPE_FROM_TRAINING_MEDIANS': 50, 'SYNTHETIC_EDGE_CASE': 73}, 'raw_status_counts': {0: 69, 1: 4, 2: 2, 3: 48}, 'rshift': 151, 'quantize_input': 28, 'argmax': 6, 'core': 21, 'raw_cases_without_exact_label_check': []}

Host C tests: ran True, passed True; compiler gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0; output: `ALL PASSED 2920`.

## Acceptance (TRAINING only)

- safety_changes_zero: PASS
- agreement_at_least_0.995: PASS
- macro_f1_drop_at_most_1pp: PASS
- golden_tests_passed: PASS

INT8 accepted: **True**. Deployment candidates: INT8 and Float32 (final choice by on-device measurement). Acceptance is measured on TRAINING rows the model was fitted on; it is not a held-out estimate and does not establish electrical safety.

Sizes (parameter storage only): {'int8_weights_bytes': 208, 'int32_bias_bytes': 84, 'tanh_lut_bytes': 256, 'float32_weight_and_bias_bytes': 916, 'note': 'parameter storage only; ESP32 flash, RAM and latency are not measured'}.

## Caveats

- Calibration and comparison use TRAINING rows only; the model was fitted on these rows.
- OPERATION was not used. INT8 behaviour on held-out or field data is not measured.
- Prediction agreement does not establish electrical safety; OPEN/SHORT outputs stay unconfirmed ML predictions and no relay isolation is implemented.
- No ESP32 memory, latency or energy result is claimed.


## Step 7: Download the results
Downloads `quant_out.zip` (INT8 parameters, manifest, generated C headers, golden vectors, report). **Send it back for review.** Nothing is committed until it is reviewed and approved.

In [9]:
# STEP 7: Zip and download
import shutil
zip_path = shutil.make_archive("/content/quant_out", "zip", OUT_ROOT)
files.download(zip_path)
print("Downloaded", zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded /content/quant_out.zip
